# Machine Translation

Machine translation was the original motivating task for the Transformer architecture [@transformers]. The encoder-decoder Transformer maps a source sentence to a target sentence through two coupled stacks: the encoder builds contextualized representations of the input, and the decoder generates the output token by token using both its own prior outputs (via **causal self-attention**) and the encoder's representations (via **cross-attention**). We implement this architecture from scratch and train it on German→English translation using the Multi30k dataset [@multi30k], measuring translation quality with BLEU score [@bleu]. We refer to the [previous notebook](../09-attention-transformers.html) for the encoder, scaled dot product attention, multi-head attention, and sinusoidal positional encoding.

The key addition in this notebook is the **Transformer decoder block**, which introduces a third sub-layer — cross-attention — that queries the encoder's output memory at each decoding step. At training time we use **teacher forcing**: the full target sequence (shifted right) is fed to the decoder in parallel, making training efficient. At inference we decode **autoregressively** — appending predicted tokens one at a time until an end-of-sequence token is emitted. We visualize the resulting cross-attention weights to confirm that the model learns a meaningful soft alignment between the source and target languages.

The full model follows the Pre-LN Transformer convention established in the previous notebook, where layer normalization is applied to sub-layer inputs rather than outputs. This stabilizes training and removes the need for a warmup schedule to achieve good convergence [@preln-transformer].

<br>

In [ ]:
import math
import random
import warnings
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from matplotlib_inline import backend_inline

DEVICE = (
    torch.device("cuda") if torch.cuda.is_available()
    else torch.device("mps") if torch.backends.mps.is_available()
    else torch.device("cpu")
)

RANDOM_SEED = 0
DEBUG = False
MATPLOTLIB_FORMAT = "png" if DEBUG else "svg"

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
warnings.filterwarnings("ignore")
backend_inline.set_matplotlib_formats(MATPLOTLIB_FORMAT)
print("device:", DEVICE)

## Dataset

**Data.** Multi30k [@multi30k] contains approximately 30,000 image captions paired in English and German. We use only the text, ignoring the images, making it a standard small-scale machine translation benchmark. Sentences are short (averaging around 13 tokens), which means a model can be trained to reasonable quality on a single GPU in minutes. We load the dataset from `torchtext.datasets.Multi30k` and tokenize at the word level using `torchtext.data.utils.get_tokenizer`. Vocabularies are built from the training split only. Four special tokens are reserved: `<pad>` (index 0), `<bos>` (1), `<eos>` (2), and `<unk>` (3). Words appearing fewer than 2 times in training are mapped to `<unk>`.

In [ ]:
from torchtext.datasets import Multi30k
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator

PAD_IDX, BOS_IDX, EOS_IDX, UNK_IDX = 0, 1, 2, 3
SPECIALS = ["<pad>", "<bos>", "<eos>", "<unk>"]
DATA_DIR = Path("./data").absolute()

tokenizer_de = get_tokenizer("basic_english")
tokenizer_en = get_tokenizer("basic_english")

# --- load raw splits ---
def load_split(split):
    pairs = []
    for de, en in Multi30k(root=str(DATA_DIR), split=split, language_pair=("de", "en")):
        pairs.append((de.strip(), en.strip()))
    return pairs

train_pairs = load_split("train")
valid_pairs = load_split("valid")
test_pairs  = load_split("test")

# --- build vocabularies from training data only ---
def yield_tokens(pairs, side, tokenizer):
    for de, en in pairs:
        yield tokenizer(de if side == "de" else en)

vocab_de = build_vocab_from_iterator(
    yield_tokens(train_pairs, "de", tokenizer_de),
    min_freq=2, specials=SPECIALS, special_first=True
)
vocab_en = build_vocab_from_iterator(
    yield_tokens(train_pairs, "en", tokenizer_en),
    min_freq=2, specials=SPECIALS, special_first=True
)
vocab_de.set_default_index(UNK_IDX)
vocab_en.set_default_index(UNK_IDX)

print(f"German vocab size : {len(vocab_de):,}")
print(f"English vocab size: {len(vocab_en):,}")
print(f"Train pairs : {len(train_pairs):,}")
print(f"Valid pairs : {len(valid_pairs):,}")
print(f"Test pairs  : {len(test_pairs):,}")
print()
de_ex, en_ex = train_pairs[0]
print(f"DE: {de_ex}")
print(f"EN: {en_ex}")

**Dataset and DataLoader.** We wrap the sentence pairs in a `TranslationDataset` and write a `collate_fn` that (1) tokenizes and numericizes each sentence, (2) prepends `<bos>` and appends `<eos>`, and (3) pads sequences within a batch to the same length using `nn.utils.rnn.pad_sequence`.

In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        de, en = self.pairs[idx]
        de_ids = [BOS_IDX] + vocab_de(tokenizer_de(de)) + [EOS_IDX]
        en_ids = [BOS_IDX] + vocab_en(tokenizer_en(en)) + [EOS_IDX]
        return torch.tensor(de_ids, dtype=torch.long), torch.tensor(en_ids, dtype=torch.long)


def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)
    src_padded = nn.utils.rnn.pad_sequence(src_batch, batch_first=True, padding_value=PAD_IDX)
    tgt_padded = nn.utils.rnn.pad_sequence(tgt_batch, batch_first=True, padding_value=PAD_IDX)
    return src_padded, tgt_padded


BATCH_SIZE = 128

train_loader = DataLoader(TranslationDataset(train_pairs), batch_size=BATCH_SIZE,
                          shuffle=True,  collate_fn=collate_fn, drop_last=True)
valid_loader = DataLoader(TranslationDataset(valid_pairs), batch_size=BATCH_SIZE,
                          shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(TranslationDataset(test_pairs),  batch_size=BATCH_SIZE,
                          shuffle=False, collate_fn=collate_fn)

# sanity-check shapes
src, tgt = next(iter(train_loader))
print(f"src batch shape: {src.shape}")
print(f"tgt batch shape: {tgt.shape}")

## Cross-attention

**Self-attention vs. cross-attention.** Recall that in the encoder, self-attention takes queries, keys, and values from the same source:

$$
\mathbf{Q} = \mathbf{X}\mathbf{W}_Q, \quad \mathbf{K} = \mathbf{X}\mathbf{W}_K, \quad \mathbf{V} = \mathbf{X}\mathbf{W}_V.
$$

**Cross-attention** uses two different sequences. Let $\mathbf{X}_\text{dec} \in \mathbb{R}^{T_\text{tgt} \times d_\text{model}}$ be the decoder's current hidden state and $\mathbf{H}_\text{enc} \in \mathbb{R}^{T_\text{src} \times d_\text{model}}$ be the encoder output. Then:

$$
\mathbf{Q} = \mathbf{X}_\text{dec}\mathbf{W}_Q, \quad \mathbf{K} = \mathbf{H}_\text{enc}\mathbf{W}_K, \quad \mathbf{V} = \mathbf{H}_\text{enc}\mathbf{W}_V.
$$

The attention score $A_{ij}$ measures how much decoder position $i$ attends to encoder position $j$:

$$
\mathbf{A} = \operatorname{Softmax}\!\left(\frac{\mathbf{Q}\mathbf{K}^\top}{\sqrt{d_h}}\right) \in \mathbb{R}^{T_\text{tgt} \times T_\text{src}},
$$

so the output $\mathbf{A}\mathbf{V} \in \mathbb{R}^{T_\text{tgt} \times d_\text{model}}$ is a context-weighted summary of the encoder states at each decoder position. This allows the decoder to selectively focus on the most relevant parts of the source sentence when generating each target token.

Cross-attention requires no masking of future positions — the encoder output is fixed and fully observed. The only masking applied here is a **source padding mask** that prevents attention to `<pad>` tokens in the source.

:::{.callout-note}

The `MultiHeadAttention` class we define below accepts separate `query`, `key`, `value` arguments, so it handles both self-attention and cross-attention with no modification. The only difference is which tensors we pass as `key` and `value`.

:::

## Decoder block

The **Transformer decoder block** contains three sub-layers, each wrapped with a Pre-LN residual connection:

1. **Causal self-attention** over the target sequence, masked so that position $i$ cannot attend to positions $j > i$.
2. **Cross-attention** where queries come from the self-attention output and keys/values come from the encoder memory $\mathbf{H}_\text{enc}$.
3. **Position-wise FFN** applied independently to each token embedding.

With $\mathbf{x}^{(0)} = \mathbf{X}_\text{dec}$ the decoder input, the forward pass is:

$$
\begin{aligned}
\mathbf{x}^{(1)} &= \mathbf{x}^{(0)} + \operatorname{Dropout}\!\left[\operatorname{SelfAttn}\!\left(\operatorname{LN}(\mathbf{x}^{(0)}),\; \text{mask}=\mathbf{M}_\text{causal}\right)\right] \\[0.5em]
\mathbf{x}^{(2)} &= \mathbf{x}^{(1)} + \operatorname{Dropout}\!\left[\operatorname{CrossAttn}\!\left(\operatorname{LN}(\mathbf{x}^{(1)}),\; \mathbf{H}_\text{enc},\; \text{mask}=\mathbf{M}_\text{pad}\right)\right] \\[0.5em]
\mathbf{x}^{(3)} &= \mathbf{x}^{(2)} + \operatorname{FFN}\!\left(\operatorname{LN}(\mathbf{x}^{(2)})\right).
\end{aligned}
$$

Note that layer normalization is applied to the residual stream *before* each sub-layer (Pre-LN), not after, which yields more stable gradient flow compared to the original Post-LN Transformer [@preln-transformer].

Defining the `MultiHeadAttention` module (adapted from the [previous notebook](../09-attention-transformers.html)):

In [ ]:
# adapted from NB09
class MultiHeadAttention(nn.Module):

    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.0):
        super().__init__()
        assert d_model % num_heads == 0, "num_heads must divide d_model"
        self.d_model  = d_model
        self.n_heads  = num_heads
        self.d_head   = d_model // num_heads
        self.dropout_p = dropout

        self.w_q = nn.Linear(d_model, d_model, bias=False)
        self.w_k = nn.Linear(d_model, d_model, bias=False)
        self.w_v = nn.Linear(d_model, d_model, bias=False)
        self.w_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, query, key, value, mask=None, return_attn=False):
        """query: (B, Tq, d), key/value: (B, Tk, d), mask: broadcastable bool."""
        B, Tq = query.shape[:2]
        Tk    = key.shape[1]

        q = self.w_q(query).view(B, Tq, self.n_heads, self.d_head).transpose(1, 2)  # (B, H, Tq, d_h)
        k = self.w_k(key  ).view(B, Tk, self.n_heads, self.d_head).transpose(1, 2)  # (B, H, Tk, d_h)
        v = self.w_v(value).view(B, Tk, self.n_heads, self.d_head).transpose(1, 2)  # (B, H, Tk, d_h)

        if mask is not None and mask.ndim == 2:
            mask = mask.unsqueeze(0).unsqueeze(0)  # (1, 1, Tq, Tk)
        elif mask is not None and mask.ndim == 3:
            mask = mask.unsqueeze(1)               # (B, 1, Tq, Tk)

        dropout_p = self.dropout_p if self.training else 0.0
        head = F.scaled_dot_product_attention(q, k, v, mask, dropout_p=dropout_p)  # (B, H, Tq, d_h)

        out = head.permute(0, 2, 1, 3).reshape(B, Tq, self.d_model)   # (B, Tq, d_model)
        out = self.w_o(out)

        if return_attn:
            # recompute weights without dropout for visualization
            scale = math.sqrt(self.d_head)
            scores = (q @ k.transpose(-2, -1)) / scale                # (B, H, Tq, Tk)
            if mask is not None:
                scores = scores.masked_fill(mask == 0, float("-inf"))
            attn_w = F.softmax(scores, dim=-1)
            return out, attn_w

        return out

Defining the `DecoderBlock` with numbered annotations:

In [ ]:
class DecoderBlock(nn.Module):

    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.self_attn  = MultiHeadAttention(d_model, num_heads, dropout)  # <1>
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)  # <2>
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model), nn.Dropout(dropout)
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, tgt, memory, tgt_mask=None, src_pad_mask=None, return_attn=False):
        # sub-layer 1: causal self-attention over target tokens
        x = self.norm1(tgt)
        x = tgt + self.dropout(self.self_attn(x, x, x, mask=tgt_mask))    # <3>

        # sub-layer 2: cross-attention — queries from decoder, keys/values from encoder
        x2 = self.norm2(x)
        if return_attn:
            ca_out, attn_w = self.cross_attn(x2, memory, memory,           # <4>
                                             mask=src_pad_mask, return_attn=True)
        else:
            ca_out  = self.cross_attn(x2, memory, memory, mask=src_pad_mask)
            attn_w  = None
        x = x + self.dropout(ca_out)

        # sub-layer 3: position-wise FFN
        x = x + self.ffn(self.norm3(x))                                    # <5>

        if return_attn:
            return x, attn_w
        return x

1. Causal self-attention — same module as in the encoder, but receives a causal mask during the forward pass so that each target position can only attend to earlier positions.
2. Cross-attention — `query` comes from the decoder stream, `key` and `value` come from the encoder memory. This is the bridge that conditions target generation on source context.
3. Pre-LN residual: normalize the stream, apply self-attention, add back. The mask `tgt_mask` is a lower-triangular boolean matrix of shape $(T_\text{tgt}, T_\text{tgt})$.
4. The encoder memory `memory` is broadcast across all decoder layers — it is computed once and never updated during decoding.
5. The FFN processes each token embedding independently, adding non-linear capacity after cross-attention.

## Encoder and full model

**Architecture.** The complete `Seq2SeqTransformer` proceeds in two phases:

- **Encode:** source tokens are embedded, positionally encoded, and passed through $N$ encoder blocks to produce `memory` $\mathbf{H}_\text{enc} \in \mathbb{R}^{B \times T_\text{src} \times d_\text{model}}$.
- **Decode:** target tokens (shifted right, prefixed with `<bos>`) are embedded, positionally encoded, and passed through $M$ decoder blocks, each attending to `memory`. The final hidden states are projected to vocabulary logits via a linear layer.

**Masking.** Three masks are used during training:

- **Source padding mask** `src_pad_mask`: shape $(B, 1, 1, T_\text{src})$, marks `<pad>` positions in the source as `False` (blocked). Prevents encoder self-attention and decoder cross-attention from attending to padding.
- **Target causal mask** `tgt_mask`: shape $(T_\text{tgt}, T_\text{tgt})$, lower-triangular boolean matrix. Prevents the decoder from seeing future target tokens during training.
- **Target padding mask**: incorporated into `tgt_mask` by zeroing out positions where the target is `<pad>`.

At inference, the source padding mask is reused, and the causal mask is grown by one position at each decoding step.

Re-implementing `PositionalEncoding` and `EncoderBlock` for self-containedness (adapted from NB09):

In [ ]:
# adapted from NB09
class PositionalEncoding(nn.Module):

    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 5000):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        t   = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        w   = torch.exp(-(torch.arange(0, d_model, 2).float() / d_model * math.log(10000)))
        pe[:, 0::2] = torch.sin(t * w)
        pe[:, 1::2] = torch.cos(t * w)
        self.register_buffer("pe", pe.unsqueeze(0))   # (1, max_len, d_model)

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1), :])


# adapted from NB09
class EncoderBlock(nn.Module):

    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.norm1     = nn.LayerNorm(d_model)
        self.norm2     = nn.LayerNorm(d_model)
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model), nn.Dropout(dropout)
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, src_mask=None):
        x = x + self.dropout(self.self_attn(self.norm1(x), self.norm1(x), self.norm1(x), mask=src_mask))
        x = x + self.ffn(self.norm2(x))
        return x

Defining the full `Seq2SeqTransformer` with mask helpers:

In [ ]:
def make_causal_mask(sz: int, device) -> torch.BoolTensor:
    """Lower-triangular boolean mask of shape (sz, sz)."""
    return torch.tril(torch.ones(sz, sz, dtype=torch.bool, device=device))


def make_src_padding_mask(src: torch.Tensor, pad_idx: int) -> torch.BoolTensor:
    """Shape (B, 1, 1, T_src): True where token is NOT pad."""
    return (src != pad_idx).unsqueeze(1).unsqueeze(2)  # (B, 1, 1, T_src)


class Seq2SeqTransformer(nn.Module):

    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, num_heads,
                 num_enc_layers, num_dec_layers, d_ff, dropout, max_len=256):
        super().__init__()
        self.d_model = d_model

        self.src_embedding = nn.Embedding(src_vocab_size, d_model, padding_idx=PAD_IDX)   # <1>
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model, padding_idx=PAD_IDX)
        self.pos_enc = PositionalEncoding(d_model, dropout, max_len)

        self.encoder = nn.ModuleList(
            [EncoderBlock(d_model, num_heads, d_ff, dropout) for _ in range(num_enc_layers)]
        )
        self.decoder = nn.ModuleList(
            [DecoderBlock(d_model, num_heads, d_ff, dropout) for _ in range(num_dec_layers)]
        )
        self.fc_out  = nn.Linear(d_model, tgt_vocab_size)                                 # <2>
        self.norm_enc = nn.LayerNorm(d_model)
        self.norm_dec = nn.LayerNorm(d_model)
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def encode(self, src, src_pad_mask):
        x = self.pos_enc(self.src_embedding(src) * math.sqrt(self.d_model))   # <3>
        for block in self.encoder:
            x = block(x, src_mask=src_pad_mask)
        return self.norm_enc(x)                                                 # <4>

    def decode(self, tgt, memory, tgt_mask, src_pad_mask, return_attn=False):
        x = self.pos_enc(self.tgt_embedding(tgt) * math.sqrt(self.d_model))
        last_attn = None
        for i, block in enumerate(self.decoder):
            is_last = (i == len(self.decoder) - 1)
            if return_attn and is_last:
                x, last_attn = block(x, memory, tgt_mask=tgt_mask,
                                     src_pad_mask=src_pad_mask, return_attn=True)
            else:
                x = block(x, memory, tgt_mask=tgt_mask, src_pad_mask=src_pad_mask)
        x = self.norm_dec(x)
        if return_attn:
            return x, last_attn
        return x

    def forward(self, src, tgt, return_attn=False):                            # <5>
        src_pad_mask  = make_src_padding_mask(src, PAD_IDX).to(src.device)
        T_tgt         = tgt.size(1)
        tgt_causal    = make_causal_mask(T_tgt, src.device)                    # <6>

        memory = self.encode(src, src_pad_mask)
        if return_attn:
            dec_out, attn = self.decode(tgt, memory, tgt_causal, src_pad_mask, return_attn=True)
            return self.fc_out(dec_out), attn
        dec_out = self.decode(tgt, memory, tgt_causal, src_pad_mask)
        return self.fc_out(dec_out)

1. Separate embedding tables for source and target languages. `padding_idx=PAD_IDX` zeroes the gradient for padding tokens so the padding embedding remains at the origin.
2. The output projection maps each hidden state to a distribution over the target vocabulary. It shares no weights with `tgt_embedding` for simplicity, though weight tying is common in practice.
3. Following [@transformers], embeddings are scaled by $\sqrt{d_\text{model}}$ before adding positional encoding, so the signal-to-noise ratio of positional information remains consistent across model sizes.
4. A final layer norm is applied after the last encoder block (Pre-LN convention: each block normalizes its own inputs, so the final output of the stack goes through an extra norm).
5. During training `src` is a padded source batch of shape $(B, T_\text{src})$; `tgt` is the padded target batch of shape $(B, T_\text{tgt})$, typically the reference shifted right (prefixed with `<bos>` and truncated before `<eos>`).
6. The causal mask is square with size $T_\text{tgt}$. Each row $i$ allows attention only to columns $j \leq i$, enforcing left-to-right generation.

Instantiating a small model and checking the parameter count:

In [ ]:
D_MODEL       = 256
NUM_HEADS     = 8
NUM_ENC_LAYERS = 3
NUM_DEC_LAYERS = 3
D_FF          = 512
DROPOUT       = 0.1
MAX_LEN       = 256

model = Seq2SeqTransformer(
    src_vocab_size=len(vocab_de),
    tgt_vocab_size=len(vocab_en),
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    num_enc_layers=NUM_ENC_LAYERS,
    num_dec_layers=NUM_DEC_LAYERS,
    d_ff=D_FF,
    dropout=DROPOUT,
    max_len=MAX_LEN,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters: {n_params:,}")

# shape check
src_t = src[:4].to(DEVICE)
tgt_t = tgt[:4, :-1].to(DEVICE)   # drop last token (shifted right)
logits = model(src_t, tgt_t)
print(f"Logits shape: {logits.shape}")

## Training

**Training setup.** During training we use **teacher forcing**: the decoder receives the full target sequence shifted right (with `<bos>` prepended) and predicts all target tokens in parallel. This is equivalent to conditioning each position $i$ on the ground-truth prefix rather than on the model's own previous predictions. The training objective is cross-entropy loss with `ignore_index=PAD_IDX` and label smoothing $\varepsilon = 0.1$, which discourages the model from becoming overconfident and has been shown to improve BLEU scores [@transformers].

**Optimizer and schedule.** We use Adam with the **Noam learning rate schedule** from [@transformers]:

$$
lr(\text{step}) = d_\text{model}^{-0.5} \cdot \min\!\left(\text{step}^{-0.5},\; \text{step} \cdot \text{warmup}^{-1.5}\right). \qquad
$$

During the first `warmup` steps the learning rate increases linearly; afterwards it decays proportionally to $\text{step}^{-0.5}$. The warmup prevents large gradient updates early in training when the model weights are still poorly initialized.

Defining the Noam scheduler and training utilities:

In [ ]:
class NoamScheduler:
    """Noam learning rate schedule from 'Attention Is All You Need'."""

    def __init__(self, optimizer, d_model: int, warmup_steps: int = 4000):
        self.optimizer     = optimizer
        self.d_model       = d_model
        self.warmup_steps  = warmup_steps
        self._step         = 0

    def step(self):
        self._step += 1
        lr = self._compute_lr()
        for pg in self.optimizer.param_groups:
            pg["lr"] = lr

    def _compute_lr(self):
        s, w = self._step, self.warmup_steps
        return self.d_model ** (-0.5) * min(s ** (-0.5), s * w ** (-1.5))


def train_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss = 0.0
    for src, tgt in loader:
        src, tgt = src.to(device), tgt.to(device)
        tgt_in  = tgt[:, :-1]   # decoder input:  <bos> w1 w2 ... w_{T-1}
        tgt_out = tgt[:, 1:]    # targets:         w1   w2 ... w_{T-1} <eos>

        logits = model(src, tgt_in)                             # (B, T-1, V)
        loss   = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    for src, tgt in loader:
        src, tgt = src.to(device), tgt.to(device)
        tgt_in  = tgt[:, :-1]
        tgt_out = tgt[:, 1:]
        logits  = model(src, tgt_in)
        loss    = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))
        total_loss += loss.item()
    return total_loss / len(loader)

Running the training loop for 15 epochs:

In [ ]:
#| output: false
NUM_EPOCHS   = 15
WARMUP_STEPS = 2000
LABEL_SMOOTH = 0.1

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX, label_smoothing=LABEL_SMOOTH)
optimizer = torch.optim.Adam(model.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9)
scheduler = NoamScheduler(optimizer, d_model=D_MODEL, warmup_steps=WARMUP_STEPS)

train_losses, valid_losses = [], []

for epoch in range(1, NUM_EPOCHS + 1):
    tr = train_epoch(model, train_loader, optimizer, scheduler, criterion, DEVICE)
    va = evaluate(model, valid_loader, criterion, DEVICE)
    train_losses.append(tr)
    valid_losses.append(va)
    print(f"Epoch {epoch:02d} | train loss {tr:.4f} | valid loss {va:.4f}")

Plotting training and validation loss curves:

In [ ]:
#| code-fold: true
#| label: fig-loss
#| fig-cap: "Training and validation cross-entropy loss over 15 epochs with the Noam learning rate schedule."
epochs = range(1, NUM_EPOCHS + 1)
plt.figure(figsize=(7, 3.5))
plt.plot(epochs, train_losses, color="C0", label="train")
plt.plot(epochs, valid_losses, color="C1", label="valid")
plt.xlabel("epoch")
plt.ylabel("cross-entropy loss")
plt.xticks(epochs)
plt.grid(linestyle="dotted", alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show();

**Figure.** Training and validation loss decrease steadily over 15 epochs. The small gap between curves indicates limited overfitting, expected given label smoothing and dropout regularization.

## Greedy decoding

At inference we cannot use teacher forcing because the reference translation is unavailable. Instead we decode **autoregressively**: (1) encode the source sequence once to produce `memory`; (2) initialize the decoder input as `[<bos>]`; (3) at each step, pass the growing decoder input through the full decoder, take the $\operatorname{argmax}$ of the last-position logits to obtain the next token; (4) append the predicted token and repeat. We stop when `<eos>` is emitted or the maximum length is reached. This greedy decoding is $O(T_\text{tgt})$ sequential forward passes through the decoder — exactly one per output token.

:::{.callout-note}

Greedy decoding is simple and fast but suboptimal: at each step it commits to the single most probable token without considering globally better alternatives. **Beam search** maintains $k$ candidate sequences simultaneously and generally yields higher BLEU scores, at the cost of $k$ times more computation.

:::

Implementing the `greedy_decode` function:

In [ ]:
@torch.inference_mode()
def greedy_decode(model, src_sentence: str, max_len: int = 50) -> str:
    """Translate a single German sentence to English using greedy decoding."""
    model.eval()
    tokens    = [BOS_IDX] + vocab_de(tokenizer_de(src_sentence)) + [EOS_IDX]
    src       = torch.tensor(tokens, dtype=torch.long).unsqueeze(0).to(DEVICE)   # (1, T_src)
    src_pad   = make_src_padding_mask(src, PAD_IDX).to(DEVICE)

    memory    = model.encode(src, src_pad)                                        # encode once
    dec_input = torch.tensor([[BOS_IDX]], dtype=torch.long, device=DEVICE)        # start token

    for _ in range(max_len):
        T = dec_input.size(1)
        causal = make_causal_mask(T, DEVICE)
        dec_out = model.decode(dec_input, memory, causal, src_pad)                # (1, T, d_model)
        dec_out = model.norm_dec(dec_out) if False else dec_out  # norm_dec applied inside decode
        logits  = model.fc_out(dec_out[:, -1, :])                                 # (1, V)
        next_id = logits.argmax(-1).item()
        dec_input = torch.cat(
            [dec_input, torch.tensor([[next_id]], device=DEVICE)], dim=1
        )
        if next_id == EOS_IDX:
            break

    # detokenize: drop <bos> and <eos>
    itos = vocab_en.get_itos()
    ids  = dec_input[0, 1:].tolist()
    words = [itos[i] for i in ids if i not in (BOS_IDX, EOS_IDX, PAD_IDX)]
    return " ".join(words)

Showing example translations from the test set:

In [ ]:
print(f"{'German source':<45}  {'Predicted (EN)':<40}  {'Reference (EN)'}")
print("-" * 130)
for de_sent, en_ref in test_pairs[:8]:
    pred = greedy_decode(model, de_sent)
    print(f"{de_sent[:44]:<45}  {pred[:39]:<40}  {en_ref[:59]}")

## BLEU score

**BLEU** (Bilingual Evaluation Understudy) [@bleu] measures $n$-gram precision between a hypothesis and a reference translation. For order $n$, let $p_n$ be the clipped fraction of $n$-grams in the hypothesis that appear in the reference. The BLEU score is:

$$
\text{BLEU} = BP \cdot \exp\!\left(\sum_{n=1}^{N} w_n \log p_n\right), \qquad BP = \min\!\left(1,\; e^{1 - r/c}\right),
$$

where $c$ is the total hypothesis length, $r$ is the total reference length, and $w_n = 1/N$ uniformly. The **brevity penalty** $BP$ discourages generating very short translations that achieve high precision trivially. BLEU-4 ($N = 4$) is the standard for MT evaluation; scores above 25 are generally considered reasonable for small-dataset systems.

:::{.callout-note}

BLEU is a corpus-level metric: it aggregates $n$-gram statistics over the entire test set rather than averaging per-sentence scores. This matters because short sentences can easily achieve high precision on lower-order $n$-grams, inflating sentence-level averages.

:::

Computing corpus BLEU-4 on the test set:

In [ ]:
from collections import Counter

def ngram_counts(tokens, n):
    return Counter(tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1))


def bleu4(hypotheses, references):
    """Corpus BLEU-4 (uniform weights, brevity penalty)."""
    clip_counts = [0] * 4
    total_counts = [0] * 4
    hyp_len = ref_len = 0

    for hyp, ref in zip(hypotheses, references):
        hyp_toks = hyp.split()
        ref_toks = ref.split()
        hyp_len += len(hyp_toks)
        ref_len += len(ref_toks)
        for n in range(1, 5):
            hyp_ng = ngram_counts(hyp_toks, n)
            ref_ng = ngram_counts(ref_toks, n)
            for gram, cnt in hyp_ng.items():
                clip_counts[n-1]  += min(cnt, ref_ng.get(gram, 0))
                total_counts[n-1] += cnt

    p_n = [
        clip_counts[n] / max(total_counts[n], 1)
        for n in range(4)
    ]
    if any(p == 0 for p in p_n):
        return 0.0

    log_avg = sum(math.log(p) for p in p_n) / 4
    bp = min(1.0, math.exp(1 - ref_len / max(hyp_len, 1)))
    return bp * math.exp(log_avg) * 100


@torch.inference_mode()
def corpus_bleu(model, pairs):
    hypotheses = [greedy_decode(model, de) for de, _ in pairs]
    references = [en for _, en in pairs]
    return bleu4(hypotheses, references)


score = corpus_bleu(model, test_pairs)
print(f"BLEU-4 on test set: {score:.2f}")

## Attention visualization

We visualize the cross-attention weights from the last decoder layer on a sample sentence. The heatmap shows which source (German) tokens each decoder step attends to when generating each target (English) token. A well-trained model should exhibit a soft **diagonal alignment** — producing English words in roughly the same order as their German counterparts — since German and English are closely related languages with similar word order.

Extracting cross-attention weights and plotting the heatmap:

In [ ]:
#| code-fold: true
#| label: fig-attn
#| fig-cap: "Cross-attention weights from the last decoder layer, averaged over all heads. Rows: generated English tokens; columns: German source tokens. Darker cells indicate higher attention weight."

@torch.inference_mode()
def translate_with_attention(model, src_sentence: str, max_len: int = 50):
    """Return (translated_tokens, src_tokens, attn_weights)."""
    model.eval()
    src_tokens = tokenizer_de(src_sentence)
    token_ids  = [BOS_IDX] + vocab_de(src_tokens) + [EOS_IDX]
    src        = torch.tensor(token_ids, dtype=torch.long).unsqueeze(0).to(DEVICE)
    src_pad    = make_src_padding_mask(src, PAD_IDX).to(DEVICE)
    memory     = model.encode(src, src_pad)

    dec_input  = torch.tensor([[BOS_IDX]], dtype=torch.long, device=DEVICE)
    all_attn   = []   # collect cross-attn at each step

    for _ in range(max_len):
        T       = dec_input.size(1)
        causal  = make_causal_mask(T, DEVICE)
        # get cross-attention from last decoder block
        dec_out, attn_w = model.decode(dec_input, memory, causal, src_pad, return_attn=True)
        # attn_w: (1, H, T, T_src) -> last position, mean over heads
        step_attn = attn_w[0, :, -1, :].mean(0)   # (T_src,)
        all_attn.append(step_attn.cpu())

        logits   = model.fc_out(dec_out[:, -1, :])
        next_id  = logits.argmax(-1).item()
        dec_input = torch.cat([dec_input,
                                torch.tensor([[next_id]], device=DEVICE)], dim=1)
        if next_id == EOS_IDX:
            break

    itos     = vocab_en.get_itos()
    pred_ids = dec_input[0, 1:].tolist()
    pred_toks = [itos[i] for i in pred_ids if i not in (BOS_IDX, EOS_IDX, PAD_IDX)]
    attn_mat  = torch.stack(all_attn[:-1]).numpy()   # drop the step that emitted <eos>
    src_display = src_tokens + ["<eos>"]
    return pred_toks, src_display, attn_mat


# pick a clean test sentence
sample_de = test_pairs[3][0]
sample_en = test_pairs[3][1]
pred_toks, src_toks, attn_mat = translate_with_attention(model, sample_de)

fig, ax = plt.subplots(figsize=(max(6, len(src_toks) * 0.6), max(4, len(pred_toks) * 0.45)))
im = ax.imshow(attn_mat, cmap="Blues", aspect="auto", vmin=0.0)
ax.set_xticks(range(len(src_toks)))
ax.set_xticklabels(src_toks, rotation=45, ha="right", fontsize=9)
ax.set_yticks(range(len(pred_toks)))
ax.set_yticklabels(pred_toks, fontsize=9)
ax.set_xlabel("source (German)")
ax.set_ylabel("generated (English)")
fig.colorbar(im, ax=ax, fraction=0.03, pad=0.04)
fig.tight_layout()
plt.show();

print(f"\nDE: {sample_de}")
print(f"EN: {' '.join(pred_toks)}")
print(f"REF: {sample_en}")

**Figure.** Cross-attention heatmap for a sample German→English translation. Each row corresponds to one generated English token; each column to one German source token. The near-diagonal structure shows that the decoder attends to the corresponding source word when generating each target word, recovering a meaningful soft alignment consistent with the two languages' shared word order.

■